In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
DESCRIBE CATALOG EXTENDED adbsmartdatatdp;

In [0]:
%sql
CREATE CATALOG smartdata;

In [0]:
%sql
SHOW GRANTS ON CATALOG adbsmartdatatdp;

In [0]:
%sql
SHOW SCHEMAS IN adbsmartdatatdp;

In [0]:
%sql
CREATE SCHEMA adbsmartdatatdp.bronze;

CREATE SCHEMA adbsmartdatatdp.silver;

CREATE SCHEMA adbsmartdatatdp.gold;

In [0]:
%sql
--====================================================
-- CATALOGO Y SCHEMAS
--====================================================

USE CATALOG adbsmartdatatdp;

USE SCHEMA bronze;

--====================================================
-- BRONZE
-- Tablas RAW
--====================================================

CREATE TABLE IF NOT EXISTS clientes_raw
(
    id_cliente INT,
    nombre STRING,
    apellido STRING,
    email STRING,
    telefono STRING,
    ciudad STRING,
    pais STRING,
    fecha_registro DATE
)
USING DELTA;

CREATE TABLE IF NOT EXISTS productos_raw
(
    id_producto INT,
    nombre_producto STRING,
    categoria STRING,
    marca STRING,
    precio DECIMAL(10,2),
    stock INT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS ventas_raw
(
    id_venta INT,
    id_cliente INT,
    id_producto INT,
    fecha_venta DATE,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    descuento DECIMAL(5,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sucursales_raw
(
    id_sucursal INT,
    nombre_sucursal STRING,
    ciudad STRING,
    pais STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS empleados_raw
(
    id_empleado INT,
    nombre STRING,
    apellido STRING,
    cargo STRING,
    id_sucursal INT
)
USING DELTA;

--====================================================
-- SILVER
-- Datos limpios
--====================================================

USE SCHEMA silver;

CREATE TABLE IF NOT EXISTS clientes
(
    id_cliente INT,
    nombre_completo STRING,
    email STRING,
    telefono STRING,
    ciudad STRING,
    pais STRING,
    fecha_registro DATE
)
USING DELTA;

CREATE TABLE IF NOT EXISTS productos
(
    id_producto INT,
    nombre_producto STRING,
    categoria STRING,
    marca STRING,
    precio DECIMAL(10,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS ventas
(
    id_venta INT,
    id_cliente INT,
    id_producto INT,
    fecha_venta DATE,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    descuento DECIMAL(5,2),
    subtotal DECIMAL(12,2),
    total DECIMAL(12,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sucursales
(
    id_sucursal INT,
    nombre_sucursal STRING,
    ciudad STRING,
    pais STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS empleados
(
    id_empleado INT,
    nombre_completo STRING,
    cargo STRING,
    id_sucursal INT
)
USING DELTA;

--====================================================
-- GOLD
-- Tablas analíticas
--====================================================

USE SCHEMA gold;

CREATE TABLE IF NOT EXISTS ventas_resumen
(
    fecha DATE,
    total_ventas DECIMAL(18,2),
    cantidad_ventas BIGINT,
    ticket_promedio DECIMAL(18,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS ventas_por_ciudad
(
    ciudad STRING,
    total_ventas DECIMAL(18,2),
    numero_ventas BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS ventas_por_categoria
(
    categoria STRING,
    total_ventas DECIMAL(18,2),
    numero_ventas BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS top_clientes
(
    id_cliente INT,
    nombre STRING,
    total_comprado DECIMAL(18,2),
    numero_compras BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS top_productos
(
    id_producto INT,
    nombre_producto STRING,
    unidades_vendidas BIGINT,
    total_vendido DECIMAL(18,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS indicadores_kpi
(
    fecha DATE,
    ingresos DECIMAL(18,2),
    clientes BIGINT,
    productos BIGINT,
    ticket_promedio DECIMAL(18,2)
)
USING DELTA;

In [0]:
%sql
USE CATALOG adbsmartdatatdp;

SHOW SCHEMAS;

In [0]:
%sql
--SHOW TABLES IN adbsmartdatatdp.bronze;

--SHOW TABLES IN adbsmartdatatdp.silver;

SHOW TABLES IN adbsmartdatatdp.gold;

In [0]:
%sql
USE CATALOG adbsmartdatatdp;

CREATE VOLUME IF NOT EXISTS bronze.files;

In [0]:
%sql
drop table if exists adbsmartdatatdp.bronze.clientes_raw;
drop table if exists adbsmartdatatdp.bronze.productos_raw;
drop table if exists adbsmartdatatdp.bronze.ventas_raw;
drop table if exists adbsmartdatatdp.bronze.sucursales_raw;
drop table if exists adbsmartdatatdp.bronze.empleados_raw;


In [0]:
%sql
drop table if exists adbsmartdatatdp.silver.clientes;
drop table if exists adbsmartdatatdp.silver.productos;
drop table if exists adbsmartdatatdp.silver.ventas;
drop table if exists adbsmartdatatdp.silver.sucursales;
drop table if exists adbsmartdatatdp.silver.empleados;

In [0]:
%sql
drop table if exists adbsmartdatatdp.gold.indicadores_kpi;
drop table if exists adbsmartdatatdp.gold.top_clientes;
drop table if exists adbsmartdatatdp.gold.top_productos;
drop table if exists adbsmartdatatdp.gold.ventas_resumen;
drop table if exists adbsmartdatatdp.gold.ventas_por_categoria;
drop table if exists adbsmartdatatdp.gold.ventas_por_ciudad;

In [0]:
%sql
--====================================================
-- SILVER
-- Clean Data Layer
--====================================================

USE CATALOG adbsmartdatatdp;
USE SCHEMA silver;

CREATE TABLE IF NOT EXISTS customers
(
    customer_id INT,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    country STRING,
    registration_date DATE
)
USING DELTA;

CREATE TABLE IF NOT EXISTS products
(
    product_id INT,
    product_name STRING,
    category STRING,
    brand STRING,
    price DECIMAL(10,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sales
(
    sale_id INT,
    customer_id INT,
    product_id INT,
    sale_date DATE,
    quantity INT,
    unit_price DECIMAL(10,2),
    discount DECIMAL(5,2),
    subtotal DECIMAL(12,2),
    total DECIMAL(12,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS branches
(
    branch_id INT,
    branch_name STRING,
    city STRING,
    country STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS employees
(
    employee_id INT,
    full_name STRING,
    position STRING,
    branch_id INT
)
USING DELTA;

--====================================================
-- GOLD
-- Business Analytics Layer
--====================================================

USE SCHEMA gold;

CREATE TABLE IF NOT EXISTS sales_summary
(
    sale_date DATE,
    total_sales DECIMAL(18,2),
    total_transactions BIGINT,
    average_ticket DECIMAL(18,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sales_by_city
(
    city STRING,
    total_sales DECIMAL(18,2),
    total_transactions BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS sales_by_category
(
    category STRING,
    total_sales DECIMAL(18,2),
    total_transactions BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS top_customers
(
    customer_id INT,
    full_name STRING,
    total_amount DECIMAL(18,2),
    total_orders BIGINT
)
USING DELTA;

CREATE TABLE IF NOT EXISTS top_products
(
    product_id INT,
    product_name STRING,
    units_sold BIGINT,
    total_sales DECIMAL(18,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS kpi_dashboard
(
    report_date DATE,
    total_revenue DECIMAL(18,2),
    total_customers BIGINT,
    total_products BIGINT,
    average_ticket DECIMAL(18,2)
)
USING DELTA;